# Nifty Total Market — Daily A/D Ratio
## 1 January 2021 to 28 September 2026

This notebook:
- downloads NSE daily CM Bhavcopy data;
- uses point-in-time Nifty 500 + Nifty Microcap 250 membership;
- calculates daily advances, declines, unchanged stocks and A/D ratio;
- exports the historical constituent intervals and final A/D CSV.

**Important:** NSE states that Nifty Total Market consists of Nifty 500 + Nifty Microcap 250. The NSE Indices page lists the Nifty Total Market series start as **1 November 2021**. The public historical-membership reconstruction used here reports reliable Microcap 250 coverage from around October 2021. The notebook therefore defaults to the official TM period. Set `USE_OFFICIAL_TM_START_ONLY = False` if you deliberately want the reconstructed early-2021 period too.


## 1. Install dependencies

In [1]:
%pip install -q -U "jugaad-data>=0.35.5" pandas numpy pyarrow requests tqdm


Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
basemap 1.3.7 requires numpy<1.25,>=1.22; python_version >= "3.8", but you have numpy 2.4.6 which is incompatible.


In [2]:
import os
import io
import time
import warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

START_DATE = pd.Timestamp("2021-01-01")
END_DATE = pd.Timestamp("2026-09-28")
OFFICIAL_TM_START = pd.Timestamp("2021-11-01")

# True: final A/D file starts on official TM series start.
# False: retain reconstructed Jan-Oct 2021 period.
USE_OFFICIAL_TM_START_ONLY = True

DATA_DIR = "Nifty_Total_Market_AD"
BHAV_DIR = os.path.join(DATA_DIR, "bhavcopy")
os.makedirs(BHAV_DIR, exist_ok=True)

print("Requested:", START_DATE.date(), "to", END_DATE.date())


Requested: 2021-01-01 to 2026-09-28


## 2. Download point-in-time Nifty 500 + Microcap 250 membership

The public reconstruction stores membership as half-open intervals: `valid_from <= date < valid_to`. This prevents today's constituents from being incorrectly applied to all historical dates.

In [3]:
MEMBERSHIP_URL = (
    "https://raw.githubusercontent.com/aditya-jha/"
    "nse-historical-membership/main/"
    "index_history/data/index_membership_history.csv"
)

membership = pd.read_csv(
    MEMBERSHIP_URL,
    parse_dates=["valid_from", "valid_to"]
)

membership.columns = [c.strip().lower() for c in membership.columns]
membership["index_name"] = membership["index_name"].astype(str).str.strip()
membership["symbol"] = membership["symbol"].astype(str).str.strip().str.upper()

TARGET_INDICES = ["Nifty 500", "Nifty Microcap 250"]

pit = membership[
    membership["index_name"].isin(TARGET_INDICES)
].copy()

pit = pit[
    (pit["valid_from"] <= END_DATE) &
    (pit["valid_to"].isna() | (pit["valid_to"] >= START_DATE))
].copy()

print("Membership rows:", len(pit))
print("Distinct symbols:", pit["symbol"].nunique())
print(pit["index_name"].value_counts())


Membership rows: 1619
Distinct symbols: 1131
index_name
Nifty Microcap 250    814
Nifty 500             805
Name: count, dtype: int64


In [4]:
pit_membership = pit[
    ["index_name", "symbol", "valid_from", "valid_to"]
].sort_values(["symbol", "valid_from"])

constituent_file = os.path.join(
    DATA_DIR,
    "nifty_total_market_constituents_intervals.csv"
)

pit_membership.to_csv(constituent_file, index=False)
print("Saved:", constituent_file)


Saved: Nifty_Total_Market_AD\nifty_total_market_constituents_intervals.csv


## 3. Initialize NSE Bhavcopy downloader

`jugaad-data` currently supports the NSE website and its historical Bhavcopy logic, including the UDiFF transition. NSE confirms that the old CM Bhavcopy was discontinued from 8 July 2024.

In [5]:
from jugaad_data.nse import NSEArchives

nse = NSEArchives()
print("NSEArchives initialized.")


NSEArchives initialized.


In [6]:
def parse_bhavcopy_text(text, dt):
    # Supports legacy NSE Bhavcopy and UDiFF column names.
    if text is None or not str(text).strip():
        return None

    raw = pd.read_csv(io.StringIO(text))
    raw.columns = [str(c).strip().upper() for c in raw.columns]

    symbol_col = next(
        (c for c in ["SYMBOL", "TCKRSYMB"] if c in raw.columns),
        None
    )
    close_col = next(
        (c for c in ["CLOSE", "CLSPRIC"] if c in raw.columns),
        None
    )
    prev_col = next(
        (c for c in ["PREVCLOSE", "PRVSCLSGPRIC"] if c in raw.columns),
        None
    )

    if symbol_col is None or close_col is None:
        raise ValueError(
            f"Could not identify SYMBOL/CLOSE columns for {dt.date()}. "
            f"Columns: {raw.columns.tolist()}"
        )

    if "SERIES" in raw.columns:
        raw = raw[
            raw["SERIES"].astype(str).str.upper().isin(["EQ", "BE"])
        ].copy()

    out = pd.DataFrame({
        "DATE": pd.Timestamp(dt).normalize(),
        "SYMBOL": raw[symbol_col].astype(str).str.strip().str.upper(),
        "CLOSE": pd.to_numeric(raw[close_col], errors="coerce")
    })

    if prev_col is not None:
        out["PREV_CLOSE"] = pd.to_numeric(
            raw[prev_col], errors="coerce"
        )
    else:
        out["PREV_CLOSE"] = np.nan

    out = out.dropna(subset=["SYMBOL", "CLOSE"])
    out = out.drop_duplicates(["DATE", "SYMBOL"], keep="last")

    return out


## 4. Test one date before downloading 5+ years

Run this cell first. If it fails, the issue is NSE connectivity/download access rather than `pd.concat`.


In [7]:
TEST_DATE = pd.Timestamp("2021-01-04")

try:
    test_text = nse.bhavcopy_raw(TEST_DATE.date())
    test_df = parse_bhavcopy_text(test_text, TEST_DATE)

    if test_df is None or test_df.empty:
        raise RuntimeError("NSE returned no usable rows.")

    print("TEST SUCCESS")
    print("Rows:", len(test_df))
    display(test_df.head())

except Exception as e:
    print("TEST FAILED")
    print(type(e).__name__, ":", e)
    raise


TEST SUCCESS
Rows: 1706


,DATE,SYMBOL,CLOSE,PREV_CLOSE
0,2021-01-04,20MICRONS,39.80,40.05
1,2021-01-04,21STCENMGM,10.60,10.65
2,2021-01-04,3IINFOTECH,7.05,6.75
3,2021-01-04,3MINDIA,21154.65,21127.60
4,2021-01-04,3PLAND,11.35,11.20


## 5. Download all NSE Bhavcopies

Each successful day is cached as Parquet, so rerunning the notebook resumes instead of downloading everything again.

In [8]:
all_days = pd.date_range(START_DATE, END_DATE, freq="B")

successful_days = []
failed_days = []

USE_CACHE = True

for dt in tqdm(all_days, desc="Downloading NSE Bhavcopies"):
    cache_file = os.path.join(BHAV_DIR, f"bhav_{dt:%Y%m%d}.parquet")

    try:
        if USE_CACHE and os.path.exists(cache_file):
            df = pd.read_parquet(cache_file)
        else:
            text = nse.bhavcopy_raw(dt.date())
            df = parse_bhavcopy_text(text, dt)

            if df is None or df.empty:
                raise ValueError("No usable equity rows.")

            df.to_parquet(cache_file, index=False)

        successful_days.append(df)

    except Exception as e:
        failed_days.append({
            "DATE": dt,
            "ERROR": f"{type(e).__name__}: {e}"
        })

    time.sleep(0.15)

print("Successful:", len(successful_days))
print("Failed:", len(failed_days))

if len(successful_days) == 0:
    raise RuntimeError(
        "ZERO Bhavcopies downloaded. Run the one-day test above and inspect its error."
    )


Successful: 1284
Failed: 213


In [9]:
bhavcopy = pd.concat(successful_days, ignore_index=True)

bhavcopy["DATE"] = pd.to_datetime(bhavcopy["DATE"])
bhavcopy["SYMBOL"] = bhavcopy["SYMBOL"].astype(str).str.strip().str.upper()
bhavcopy = bhavcopy.sort_values(["DATE", "SYMBOL"]).reset_index(drop=True)

bhavcopy_file = os.path.join(
    DATA_DIR,
    "nse_equity_bhavcopy_2021_2026.parquet"
)

bhavcopy.to_parquet(bhavcopy_file, index=False)

print("Rows:", len(bhavcopy))
print("Date range:", bhavcopy["DATE"].min(), "to", bhavcopy["DATE"].max())
print("Saved:", bhavcopy_file)


Rows: 3146508
Date range: 2021-01-01 00:00:00 to 2026-09-28 00:00:00
Saved: Nifty_Total_Market_AD\nse_equity_bhavcopy_2021_2026.parquet


## 6. Apply point-in-time Nifty Total Market membership

In [10]:
tm_intervals = (
    pit_membership
    .drop_duplicates(["symbol", "valid_from", "valid_to"])
    [["symbol", "valid_from", "valid_to"]]
    .sort_values(["symbol", "valid_from"])
)

price = bhavcopy[
    bhavcopy["SYMBOL"].isin(set(tm_intervals["symbol"]))
].copy()

price = price.merge(
    tm_intervals,
    left_on="SYMBOL",
    right_on="symbol",
    how="inner"
)

price = price[
    (price["DATE"] >= price["valid_from"]) &
    (price["valid_to"].isna() | (price["DATE"] < price["valid_to"]))
].copy()

price.drop(columns=["symbol"], inplace=True)
price = price.drop_duplicates(["DATE", "SYMBOL"], keep="last")

print("Point-in-time rows:", len(price))
print("Trading dates:", price["DATE"].nunique())


Point-in-time rows: 933922
Trading dates: 1284


## 7. Classify Advance / Decline / Unchanged

The calculation uses NSE's `PREVCLOSE` where available. That is preferable to blindly shifting rows because a stock can be missing from a particular day's file.

In [11]:
price["CLOSE"] = pd.to_numeric(price["CLOSE"], errors="coerce")
price["PREV_CLOSE"] = pd.to_numeric(price["PREV_CLOSE"], errors="coerce")
price = price.dropna(subset=["CLOSE"])

price["MOVEMENT"] = np.select(
    [
        price["PREV_CLOSE"].notna() & (price["CLOSE"] > price["PREV_CLOSE"]),
        price["PREV_CLOSE"].notna() & (price["CLOSE"] < price["PREV_CLOSE"]),
        price["PREV_CLOSE"].notna() & (price["CLOSE"] == price["PREV_CLOSE"])
    ],
    ["ADVANCE", "DECLINE", "UNCHANGED"],
    default="NO_PREVIOUS_CLOSE"
)

print(price["MOVEMENT"].value_counts(dropna=False))


MOVEMENT
DECLINE      483591
ADVANCE      445556
UNCHANGED      4775
Name: count, dtype: int64


## 8. Calculate daily A/D statistics

In [12]:
daily_ad = (
    price
    .groupby("DATE")
    .agg(
        ADVANCES=("MOVEMENT", lambda x: (x == "ADVANCE").sum()),
        DECLINES=("MOVEMENT", lambda x: (x == "DECLINE").sum()),
        UNCHANGED=("MOVEMENT", lambda x: (x == "UNCHANGED").sum()),
        NO_PREVIOUS_CLOSE=("MOVEMENT", lambda x: (x == "NO_PREVIOUS_CLOSE").sum()),
        STOCKS_TRADED=("SYMBOL", "nunique")
    )
    .reset_index()
)

daily_ad["AD_RATIO"] = (
    daily_ad["ADVANCES"] /
    daily_ad["DECLINES"].replace(0, np.nan)
)

daily_ad["AD_BREADTH"] = (
    (daily_ad["ADVANCES"] - daily_ad["DECLINES"]) /
    (daily_ad["ADVANCES"] + daily_ad["DECLINES"])
).replace([np.inf, -np.inf], np.nan)

daily_ad["LOG_AD_RATIO"] = np.log(daily_ad["AD_RATIO"])

rolling_mean = daily_ad["LOG_AD_RATIO"].rolling(60, min_periods=20).mean()
rolling_std = daily_ad["LOG_AD_RATIO"].rolling(60, min_periods=20).std()

daily_ad["LOG_AD_RATIO_Z60"] = (
    (daily_ad["LOG_AD_RATIO"] - rolling_mean) /
    rolling_std
)

daily_ad = daily_ad.sort_values("DATE").reset_index(drop=True)

display(daily_ad.head())


,DATE,ADVANCES,DECLINES,UNCHANGED,NO_PREVIOUS_CLOSE,STOCKS_TRADED,AD_RATIO,AD_BREADTH,LOG_AD_RATIO,LOG_AD_RATIO_Z60
0,2021-01-01,494,179,11,0,684,2.759777,0.468053,1.015150,NaN
1,2021-01-04,489,186,9,0,684,2.629032,0.448889,0.966616,NaN
2,2021-01-05,347,329,8,0,684,1.054711,0.026627,0.053267,NaN
3,2021-01-06,264,416,4,0,684,0.634615,-0.223529,-0.454736,NaN
4,2021-01-08,417,259,8,0,684,1.610039,0.233728,0.476258,NaN


## 9. Restrict to official Nifty Total Market series start

In [13]:
if USE_OFFICIAL_TM_START_ONLY:
    final_ad = daily_ad[daily_ad["DATE"] >= OFFICIAL_TM_START].copy()
else:
    final_ad = daily_ad.copy()

final_ad = final_ad[
    (final_ad["DATE"] >= START_DATE) &
    (final_ad["DATE"] <= END_DATE)
].copy()

print("Final range:", final_ad["DATE"].min(), "to", final_ad["DATE"].max())
print("Trading days:", len(final_ad))


Final range: 2021-11-01 00:00:00 to 2026-09-28 00:00:00
Trading days: 1113


## 10. Data-quality checks

In [14]:
final_ad["COVERAGE_PCT"] = final_ad["STOCKS_TRADED"] / 750 * 100

display(final_ad.describe())

low_coverage = final_ad[final_ad["STOCKS_TRADED"] < 650]

print("Dates with <650 stocks:", len(low_coverage))
display(low_coverage.head(20))


,DATE,ADVANCES,DECLINES,UNCHANGED,NO_PREVIOUS_CLOSE,STOCKS_TRADED,AD_RATIO,AD_BREADTH,LOG_AD_RATIO,LOG_AD_RATIO_Z60,COVERAGE_PCT
count,1113,1113.000000,1113.000000,1113.000000,1113.0,1113.000000,1113.000000,1113.000000,1113.000000,1113.000000,1113.000000
mean,2024-05-24 08:00:00,349.362084,378.390836,3.446541,0.0,731.199461,1.409669,-0.039555,-0.115834,-0.015840,97.493261
min,2021-11-01 00:00:00,8.000000,35.000000,0.000000,0.0,712.000000,0.011299,-0.977654,-4.483003,-3.894087,94.933333
25%,2023-03-01 00:00:00,248.000000,274.000000,1.000000,0.0,724.000000,0.513742,-0.321229,-0.666034,-0.606895,96.533333
50%,2024-06-28 00:00:00,358.000000,369.000000,3.000000,0.0,729.000000,0.975342,-0.012483,-0.024967,0.045060,97.200000
75%,2025-08-11 00:00:00,451.000000,478.000000,5.000000,0.0,737.000000,1.650558,0.245442,0.501113,0.613176,98.266667
max,2026-09-28 00:00:00,701.000000,722.000000,16.000000,0.0,749.000000,19.457143,0.902235,2.968214,2.469832,99.866667
std,NaN,145.267234,145.885059,3.063928,0.0,9.865905,1.738806,0.398647,0.996325,0.998088,1.315454


Dates with <650 stocks: 0


,DATE,ADVANCES,DECLINES,UNCHANGED,NO_PREVIOUS_CLOSE,STOCKS_TRADED,AD_RATIO,AD_BREADTH,LOG_AD_RATIO,LOG_AD_RATIO_Z60,COVERAGE_PCT


## 11. Save outputs

In [15]:
FINAL_AD_FILE = os.path.join(
    DATA_DIR,
    "Nifty_Total_Market_AD_Ratio_2021_2026.csv"
)

QUALITY_FILE = os.path.join(
    DATA_DIR,
    "Nifty_Total_Market_AD_Ratio_quality.csv"
)

STOCK_LEVEL_FILE = os.path.join(
    DATA_DIR,
    "Nifty_Total_Market_stock_level_movements.parquet"
)

final_ad.to_csv(FINAL_AD_FILE, index=False)
final_ad.to_csv(QUALITY_FILE, index=False)

price[
    ["DATE", "SYMBOL", "CLOSE", "PREV_CLOSE", "MOVEMENT"]
].to_parquet(STOCK_LEVEL_FILE, index=False)

print("Saved:")
print(FINAL_AD_FILE)
print(QUALITY_FILE)
print(STOCK_LEVEL_FILE)


Saved:
Nifty_Total_Market_AD\Nifty_Total_Market_AD_Ratio_2021_2026.csv
Nifty_Total_Market_AD\Nifty_Total_Market_AD_Ratio_quality.csv
Nifty_Total_Market_AD\Nifty_Total_Market_stock_level_movements.parquet


## 12. Final result

In [16]:
display(final_ad.tail(20))

print("Columns:")
print(final_ad.columns.tolist())


,DATE,ADVANCES,DECLINES,UNCHANGED,NO_PREVIOUS_CLOSE,STOCKS_TRADED,AD_RATIO,AD_BREADTH,LOG_AD_RATIO,LOG_AD_RATIO_Z60,COVERAGE_PCT
1264,2026-08-31,286,460,1,0,747,0.621739,-0.233244,-0.475235,-0.475434,99.600000
1265,2026-09-01,282,459,6,0,747,0.614379,-0.238866,-0.487143,-0.534773,99.600000
1266,2026-09-02,238,507,2,0,747,0.469428,-0.361074,-0.756240,-0.836439,99.600000
1267,2026-09-03,455,289,3,0,747,1.574394,0.223118,0.453871,0.643022,99.600000
1268,2026-09-04,375,369,3,0,747,1.016260,0.008065,0.016129,0.064802,99.600000
1269,2026-09-07,280,465,2,0,747,0.602151,-0.248322,-0.507248,-0.599100,99.600000
1270,2026-09-08,333,408,6,0,747,0.816176,-0.101215,-0.203125,-0.140954,99.600000
1271,2026-09-09,248,496,3,0,747,0.500000,-0.333333,-0.693147,-0.825902,99.600000
1272,2026-09-10,282,460,5,0,747,0.613043,-0.239892,-0.489319,-0.510124,99.600000
1273,2026-09-11,246,496,5,0,747,0.495968,-0.336927,-0.701244,-0.791425,99.600000


Columns:
['DATE', 'ADVANCES', 'DECLINES', 'UNCHANGED', 'NO_PREVIOUS_CLOSE', 'STOCKS_TRADED', 'AD_RATIO', 'AD_BREADTH', 'LOG_AD_RATIO', 'LOG_AD_RATIO_Z60', 'COVERAGE_PCT']


## 13. HMM recommendation

For your market-regime HMM, retain:

- `AD_RATIO` — intuitive raw ratio.
- `AD_BREADTH` — bounded breadth measure.
- `LOG_AD_RATIO` — symmetric around zero.
- `LOG_AD_RATIO_Z60` — rolling standardized breadth.

When training/testing the HMM, fit any scaler on the training period only; do not standardize the entire 2021–2026 sample before the out-of-sample split.
